In [1]:
import ast
import json
import re
import requests

import pandas as pd
import numpy as np

from bs4 import BeautifulSoup

from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from utils import extraer_informacion, obtener_urls, _safe_eval, parse_nested_tecnocasa, _to_m, poi_features_tecnocasa, final_clean_tecnocasa

pd.set_option('display.max_columns', 500)

url = 'https://www.tecnocasa.es/venta/piso/mapa.html/pag-1?view=41.18051487737182,-2.109104143945018,39.64144152264612,-5.319858538475728&zoom=9&sort=Aggiornamento|DESC' # Mapa de la Comunidad de Madrid y alrededores
df = pd.read_csv('../data/pisos_madrid.csv', sep ='|') # NO TENEMOS QUE HACERLO DESPUES DEL SCRAPING ? 

# Extracción de datos

In [4]:
response = requests.get(url, timeout=10)
soup = BeautifulSoup(response.text, "html.parser")
ultima = soup.find('a', string='>>')
max_pages = int(re.findall(r'pag-(\d+)', str(ultima))[0])

url_splited = url.split('pag-1')
data = []
for i in range(1, max_pages+1):
    subdata = obtener_urls(f'pag-{i}'.join(url_splited), df)
    data.extend(subdata)

    if len(subdata) < 15:
        break

df = pd.concat([pd.DataFrame(data), df])
df.to_csv('../data/pisos_madrid.csv', sep='|', index=False)

Buscando pisos en la página https://www.tecnocasa.es/venta/piso/mapa.html/pag-1?view=41.18051487737182,-2.109104143945018,39.64144152264612,-5.319858538475728&zoom=9&sort=Aggiornamento|DESC ...
Ya lo tengo: https://www.tecnocasa.es/venta/piso/madrid/madrid/649791.html


# Pretratamiento de datos

In [5]:
# A partir de este punto sería interesante integrar un pipeline, para que si entra un nuevo piso todo se ejecute en el orden adecuado.

In [6]:
DROP_COLS = ['url', 'features', 'descripcion', 'precio', 'media', 'points_of_interest',
             'energy_data', 'transporte_publico', 'escuelas', 'farmacias', 'hospitales',
             'supermercados', 'tiendas', 'bares', 'restaurantes']
drop_step = FunctionTransformer(
    lambda df: df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore")
)

In [7]:
pipe_features = Pipeline([
    ("parse_nested", FunctionTransformer(parse_nested_tecnocasa)),        
    ("poi_features", FunctionTransformer(poi_features_tecnocasa)),          
    ("final_clean", FunctionTransformer(final_clean_tecnocasa)),   
    ("drop",drop_step)
])

df_feat = pipe_features.fit_transform(df)

In [8]:
df_feat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1209 entries, 0 to 1208
Data columns (total 37 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   dormitorios            1098 non-null   float64
 1   superficie_m2          1209 non-null   float64
 2   baños                  1112 non-null   float64
 3   latitud                1209 non-null   float64
 4   longitud               1209 non-null   float64
 5   planta                 762 non-null    object 
 6   aire_acondicionado     452 non-null    object 
 7   ascensor               1209 non-null   int64  
 8   calefaccion            669 non-null    object 
 9   categoria              1209 non-null   object 
 10  año_construccion       1209 non-null   int64  
 11  precio_euros           1209 non-null   int64  
 12  hipoteca               1209 non-null   int64  
 13  planos                 1209 non-null   int64  
 14  realistico             1209 non-null   int64  
 15  foto

In [24]:
# Separar en X_train, X_test, y_train , y_test
# Otro pipeline

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('nombre', funcion, ['columnas']),
    ],
    remainder="drop"
)

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_sel),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_sel),
    ],
    remainder="drop"
)